# NB01 — Linear Regression: Statistical Intuition

> **StatQuest principle: build the picture first, formula second.**

---

## The main ideas are:

1. You have two variables and you suspect one can **predict** the other
2. You fit a straight line through the data
3. The line is chosen to **minimise prediction errors** (residuals)
4. OLS = Ordinary Least Squares = the method that finds that unique best line

---

### Josh Starmer's framing (StatQuest):
> *"We want to fit a line to data. But which line? The one that makes the vertical distances between the points and the line as small as possible. And we square those distances so big errors are penalised more."*


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    """Draw a horizontal flow diagram.  steps = list of strings."""
    n = len(steps)
    default_colors = [
        '#1565C0','#2E7D32','#E65100','#6A1B9A',
        '#00695C','#AD1457','#37474F','#4E342E',
        '#0277BD','#558B2F',
    ]
    colors = (colors or default_colors[:n]) + default_colors
    notes  = notes or [''] * n

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n * 3.1)
    ax.set_ylim(-1.2, 2.4)
    ax.axis('off')

    bw, bh = 2.6, 1.3
    for i, (step, color, note) in enumerate(zip(steps, colors, notes)):
        x = i * 3.1
        box = FancyBboxPatch((x, 0.2), bw, bh,
                             boxstyle="round,pad=0.12",
                             facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.90)
        ax.add_patch(box)
        ax.text(x + bw/2, 0.2 + bh/2, step,
                ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold', multialignment='center')
        if note:
            ax.text(x + bw/2, 0.02, note,
                    ha='center', va='top', fontsize=7, color='#555', style='italic')
        if i < n - 1:
            ax.annotate('',
                xy=(x + bw + 0.38, 0.2 + bh/2),
                xytext=(x + bw + 0.08, 0.2 + bh/2),
                arrowprops=dict(arrowstyle='->', color='#444', lw=2.2))

    ax.set_title(title, fontsize=11, fontweight='bold', pad=6, color='#222')
    plt.tight_layout(pad=0.4)
    plt.show()

flow_diagram(
    steps=[
        'Collect\n(x, y) pairs',
        'Scatter plot:\nspot the trend',
        'Try many\nlines',
        'Measure each\nline's SSR',
        'Pick the line\nwith min SSR',
        'Use it to\npredict new y',
    ],
    notes=[
        'e.g. hours & score',
        'is it roughly linear?',
        'different b0, b1',
        'SSR = sum of\nsquared residuals',
        'that's OLS!',
        'y = b0 + b1*x',
    ],
    title='NB01 Conceptual Flow: How Linear Regression Works',
    colors=['#1565C0','#2E7D32','#E65100','#6A1B9A','#C62828','#00695C'],
)


## Step 1 — What does "fitting a line" mean?

A line has exactly **two numbers** to choose:

| Symbol | Name | Meaning |
|--------|------|---------|
| beta_0 (b0) | Intercept | predicted y when x = 0 |
| beta_1 (b1) | Slope | how much y changes for +1 unit of x |

**Equation:** `y_hat = b0 + b1 * x`

The hat on y_hat means "predicted value" (not the real y).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Study hours vs exam score — our running example
hours  = np.array([1,2,3,4,5,6,7,8,9,10], dtype=float)
scores = np.array([50,53,61,67,70,75,79,84,89,95], dtype=float)

# --- StatQuest moment: visualise the raw relationship first ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(hours, scores, s=90, color='steelblue', zorder=3, label='Observed data')
ax.set_xlabel('Study hours', fontsize=12)
ax.set_ylabel('Exam score', fontsize=12)
ax.set_title('Step 1: Look at the data — is there a linear trend?', fontsize=12)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()

print("Key question: can we draw ONE straight line that is 'close' to all points?")
print("If yes -> linear regression is appropriate.")


## Step 2 — Residuals: the core idea

For each data point the **residual** (error) is:

```
residual_i  =  y_i  -  y_hat_i   =  actual  -  predicted
```

- Positive residual: line **under**-predicted
- Negative residual: line **over**-predicted
- Zero residual: perfect prediction

**Why do we care?** Because OLS finds the line that makes ALL residuals as small as possible simultaneously.


In [ ]:
# Show two different lines and their residuals
import numpy as np
import matplotlib.pyplot as plt

def plot_line_residuals(ax, hours, scores, b0, b1, title, color):
    y_pred = b0 + b1 * hours
    ssr    = np.sum((scores - y_pred)**2)
    ax.scatter(hours, scores, s=70, color='steelblue', zorder=4)
    ax.plot(hours, y_pred, color=color, linewidth=2.5, label=f'b0={b0}, b1={b1}')
    for xi, yi, yp in zip(hours, scores, y_pred):
        ax.vlines(xi, min(yi,yp), max(yi,yp), color='tomato', linewidth=2, alpha=0.7)
    ax.set_title(f'{title}\nSSR = {ssr:.1f}', fontsize=10)
    ax.set_xlabel('Hours'); ax.set_ylabel('Score')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# True OLS line
xbar = hours.mean(); ybar = scores.mean()
b1_ols = np.sum((hours-xbar)*(scores-ybar)) / np.sum((hours-xbar)**2)
b0_ols = ybar - b1_ols * xbar

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_line_residuals(axes[0], hours, scores, 30, 7,   'Bad line (b1=7)', 'orange')
plot_line_residuals(axes[1], hours, scores, 40, 5.5, 'Better line (b1=5.5)', 'purple')
plot_line_residuals(axes[2], hours, scores, b0_ols, b1_ols, 'OLS best-fit (minimum SSR)', 'crimson')

plt.suptitle('Red lines = residuals. OLS picks the line with the SMALLEST total squared residuals.',
             fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

print(f"OLS line: score = {b0_ols:.2f} + {b1_ols:.2f} * hours")
print(f"Interpretation: each extra study hour adds {b1_ols:.2f} points on average")


## Step 3 — Why SQUARE the residuals?

**StatQuest explanation:**
- If we just summed residuals, positives and negatives cancel out — a terrible line could score zero!
- Squaring makes ALL residuals positive
- Squaring penalises **big** errors much more than small ones (2x error = 4x penalty)
- The SSR surface is a smooth bowl with exactly ONE minimum — easy to find


In [ ]:
# Visualise why squaring works — show residuals as physical squares
import numpy as np
import matplotlib.pyplot as plt

b0, b1 = b0_ols, b1_ols
y_pred = b0 + b1*hours

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: residuals as vertical lines
ax = axes[0]
ax.scatter(hours, scores, s=70, color='steelblue', zorder=4)
ax.plot(hours, y_pred, 'r-', linewidth=2.5, label='Best-fit line')
for xi, yi, yp in zip(hours, scores, y_pred):
    ax.vlines(xi, min(yi,yp), max(yi,yp), color='orange', linewidth=2.5)
ax.set_title('Residuals as vertical distances', fontsize=10)
ax.set_xlabel('Hours'); ax.set_ylabel('Score')
ax.legend(); ax.grid(alpha=0.3)

# Right: residuals as squares (area = squared residual)
ax = axes[1]
ax.scatter(hours, scores, s=70, color='steelblue', zorder=4)
ax.plot(hours, y_pred, 'r-', linewidth=2.5, label='Best-fit line')
for xi, yi, yp in zip(hours, scores, y_pred):
    res   = yi - yp
    side  = abs(res)
    scale = 0.12          # visual scale factor for x-axis
    sq_x  = [xi, xi + side*scale, xi + side*scale, xi, xi]
    sq_y  = [yp, yp, yp + res, yp + res, yp]
    ax.fill(sq_x, sq_y, alpha=0.25, color='orange')
    ax.vlines(xi, min(yi,yp), max(yi,yp), color='orange', linewidth=2)

total_area = np.sum((scores - y_pred)**2)
ax.set_title(f'Squared residuals as squares\nOLS minimises total area = {total_area:.1f}',
             fontsize=10)
ax.set_xlabel('Hours'); ax.set_ylabel('Score')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('"Least Squares" = minimize the total area of all orange squares', fontsize=12)
plt.tight_layout(); plt.show()


## Key Takeaways

| Term | StatQuest plain-English version |
|------|-------------------------------|
| Linear regression | Draw the best-fitting straight line through the data |
| b0 (intercept) | Where the line crosses the y-axis |
| b1 (slope) | How steep the line is — y change per x change |
| Residual | How far each point is from the line (vertically) |
| SSR | Total area of all squared-residual boxes |
| OLS | The method that finds the unique line with minimum SSR |

**Next: NB02 — where the formula `b1 = cov(x,y)/var(x)` comes from, derived step by step.**
